# 01. Exploración y calidad de los datos
## Análisis de sentimientos en versos de poemas

**Autor:** Luis Ariel Vázquez Piña  
**Versión del proyecto:** 2.0

## Objetivo

Explorar el conjunto de datos Poem Sentiment para comprender su
estructura, verificar el significado de sus etiquetas y evaluar
su calidad antes de entrenar los modelos.

## Alcance

En este notebook se revisarán:

- Las particiones de entrenamiento, validación y prueba.
- El mapeo de etiquetas definido en los metadatos.
- La distribución de clases por partición.
- Los valores faltantes y textos vacíos.
- Los versos duplicados y posibles coincidencias entre particiones.
- La distribución de longitud de los versos.

Se conservarán inicialmente las cuatro categorías originales.
Cualquier exclusión o transformación posterior se documentará
junto con su justificación.

In [1]:
import sys

print("Intérprete del notebook:", sys.executable)

Intérprete del notebook: /opt/anaconda3/envs/poemas-ia/bin/python


## 1. Carga del conjunto de datos

Se utiliza el conjunto de datos
`google-research-datasets/poem_sentiment`, disponible en Hugging Face.

Se mantienen sus particiones originales de entrenamiento,
validación y prueba. Los nombres de las etiquetas se consultan
directamente desde los metadatos para evitar asignaciones
manuales incorrectas.

La primera descarga requiere conexión a internet.

In [2]:
import sys
import pandas as pd
import numpy as np
import sklearn
from datasets import load_dataset

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Entorno listo")

/opt/anaconda3/envs/poemas-ia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: /opt/anaconda3/envs/poemas-ia/bin/python
Pandas: 3.0.6
NumPy: 2.5.3
Scikit-learn: 1.9.1
Entorno listo


In [3]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("google-research-datasets/poem_sentiment")

print(dataset)

etiquetas = dataset["train"].features["label"].names
print("\nEtiquetas originales:")
for indice, nombre in enumerate(etiquetas):
    print(f"{indice}: {nombre}")

Generating test split: 100%|██████████| 104/104 [00:00<00:00, 35349.08 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 104
    })
})

Etiquetas originales:
0: negative
1: positive
2: no_impact
3: mixed


In [4]:
distribucion = pd.DataFrame({
    particion: (
        pd.Series(dataset[particion]["label"])
        .value_counts()
        .reindex(range(len(etiquetas)), fill_value=0)
    )
    for particion in dataset
})

distribucion.index = etiquetas
distribucion.index.name = "Sentimiento"

display(distribucion)
display(dataset["train"].to_pandas().head())

,train,validation,test
Sentimiento,,,
negative,155,19,19
positive,133,17,16
no_impact,555,69,69
mixed,49,0,0


,id,verse_text,label
0,0,with pale blue berries. in these peaceful shad...,1
1,1,"it flows so long as falls the rain,",2
2,2,"and that is why, the lonesome day,",0
3,3,"when i peruse the conquered fame of heroes, an...",3
4,4,of inward strife for truth and liberty.,3


In [5]:
# Conservar las tres clases del experimento principal.
# Mantener los identificadores originales para evitar confusiones:
# 0 = negative, 1 = positive, 2 = no_impact

CLASES = {
    0: "negative",
    1: "positive",
    2: "no_impact",
}

datos = {
    particion: dataset[particion].to_pandas().loc[
        lambda df: df["label"].isin(CLASES)
    ].copy()
    for particion in dataset
}

train_df = datos["train"]
val_df = datos["validation"]
test_df = datos["test"]

for nombre, df in datos.items():
    assert df["verse_text"].notna().all(), f"Textos nulos en {nombre}"
    assert df["verse_text"].str.strip().ne("").all(), (
        f"Textos vacíos en {nombre}"
    )
    print(f"{nombre}: {len(df)} versos")

train: 843 versos
validation: 105 versos
test: 104 versos


In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(train_df[["verse_text"]], train_df["label"])
predicciones = baseline.predict(val_df[["verse_text"]])

print(
    "Accuracy:",
    round(accuracy_score(val_df["label"], predicciones), 4)
)
print(
    "F1 macro:",
    round(f1_score(
        val_df["label"],
        predicciones,
        labels=list(CLASES),
        average="macro",
        zero_division=0
    ), 4)
)

Accuracy: 0.6571
F1 macro: 0.2644
